# Most recent benchmarks for BRCA dataset

In [20]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variance_filtering, class_variational_selection
from bipartite_gnn.preprocessing import mrmr_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval
from bipartite_gnn.graph_building import cosine_similarity_matrix, keep_n_neighbours, dense_to_coo

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load data and make sure that everything is aligned

In [16]:
null_vals = ["NA"]

mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
meth = pl.read_csv("BRCA_PROCESSED_DATA/meth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)
cnv = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())

assert mrna.columns[1:] == meth.columns[1:] == cnv.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [17]:
# concat all data into one dataframe
mrna_X = mrna[:, 1:].to_numpy().T
meth_X = meth[:, 1:].to_numpy().T
mirna_X = mirna[:, 1:].to_numpy().T
cnv_X = cnv[:, 1:].to_numpy().T

print(mrna_X.shape, meth_X.shape, mirna_X.shape, cnv_X.shape)

# concat after preprocessing
X_list = [mrna_X] # , meth_X, mirna_X, cnv_X]

# concat before preprocessing
X = np.hstack(X_list)

(483, 18975) (483, 9317) (483, 231) (483, 24776)


In [103]:
knn_eval(X, y, select_n_features=True)

500
| KNN | 0.84 +/- 0.01 | 0.82 +/- 0.02 | 0.83 +/- 0.01 |
study.best_value=0.8294639374375246, study.best_params={'n_neighbors': 11, 'n_features': 1859}


## concat before

| Model | ACC | F1 macro | F1 weighted |
| --- | --- | --- | --- | 
| KNN (mrna) | 0.84 +/- 0.02 | 0.83 +/- 0.02 | 0.83 +/- 0.02 |
| KNN (mrna, mirna) | 0.84 +/- 0.02 | 0.83 +/- 0.03 | 0.84 +/- 0.02 |
| KNN (mrna, mirna, meth) | 0.80 +/- 0.04 | 0.78 +/- 0.06 | 0.79 +/- 0.05 |
| KNN (mrna, mirna, meth, cnv) | 0.80 +/- 0.04 | 0.78 +/- 0.05 | 0.79 +/- 0.05 |

## concat after (KNN)

| subset | weighted f1 |
| --- | --- |
| mrna | 0.83 +/- 0.01 |
| mirna | 0.68 +/- 0.04 |
| meth | 0.61 +/- 0.07 |
| cnv | 0.55 +/- 0.03 |
| mrna, mirna | 0.82 +/- 0.03 |
| mrna, mirna, meth | 0.78 +/- 0.03 |
| mrna, mirna, meth, cnv | 0.72 +/- 0.03 |

In [106]:
svm_eval(X, y, select_n_features=True, n_trials=20, n_features_preselect=10000)

Trial 1 / 20
| SVM | 0.82 +/- 0.02 | 0.82 +/- 0.02 | 0.83 +/- 0.02 |
Trial 2 / 20
| SVM | 0.83 +/- 0.02 | 0.83 +/- 0.02 | 0.83 +/- 0.02 |
Trial 3 / 20
Trial 4 / 20
Trial 5 / 20
Trial 6 / 20
Trial 7 / 20
Trial 8 / 20
Trial 9 / 20
Pruning trial
Trial 10 / 20
Trial 11 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |
Trial 12 / 20
Pruning trial
Trial 13 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
Trial 14 / 20
Pruning trial
Trial 15 / 20
Trial 16 / 20
Pruning trial
Trial 17 / 20
Pruning trial
Trial 18 / 20
Pruning trial
Trial 19 / 20
Pruning trial
Trial 20 / 20
| LIN SVM | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
study.best_value=0.834053661222406, study.best_params={'C': 0.00965373842048661, 'class_weight': 'balanced', 'n_features': 1770}


{'acc': 0.832323883161512,
 'f1_macro': 0.8283495019959709,
 'f1_weighted': 0.834053661222406,
 'acc_std': 0.009659601422776734,
 'f1_macro_std': 0.010556149693910116,
 'f1_weighted_std': 0.009406892392122062}

In [ ]:
| LIN SVM (mrna) | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
| LIN SVM (all) | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |

In [108]:
xgboost_eval(X, y, n_trials=20, n_features=5000)

0 / 20
ACC: [0.45360825 0.45360825 0.45360825 0.45833333 0.45833333]
F1M: [0.15602837 0.15602837 0.15602837 0.15714286 0.15714286]
F1W: [0.28310302 0.28310302 0.28310302 0.28809524 0.28809524]
| XGBoost | 0.46 +/- 0.00 | 0.16 +/- 0.00 | 0.29 +/- 0.00 |
1 / 20
ACC: [0.84536082 0.89690722 0.87628866 0.85416667 0.875     ]
F1M: [0.82777469 0.90148423 0.87620773 0.87072756 0.85983827]
F1W: [0.84792499 0.89770671 0.87467005 0.8508768  0.8756662 ]
| XGBoost | 0.87 +/- 0.02 | 0.87 +/- 0.02 | 0.87 +/- 0.02 |
2 / 20
Pruning trial
ACC: [0.81443299 0.79381443 0.81443299 0.80208333 0.        ]
F1M: [0.78562843 0.8141908  0.79612403 0.82675803 0.        ]
F1W: [0.81221407 0.79860159 0.81181172 0.79870123 0.        ]
3 / 20
Pruning trial
ACC: [0.82474227 0.80412371 0.81443299 0.83333333 0.        ]
F1M: [0.76158266 0.78914288 0.80304557 0.8151346  0.        ]
F1W: [0.82362017 0.7957014  0.81356722 0.83152298 0.        ]
4 / 20
Pruning trial
ACC: [0.75257732 0.70103093 0.65979381 0.71875    0.       

| XGBoost (default) | 0.80 +/- 0.02 | 0.80 +/- 0.04 | 0.80 +/- 0.03 |  
| XGBoost (tuned, mrna only) | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |  
| XGBoost (tuned, all) | 0.88 +/- 0.03 | 0.87 +/- 0.04 | 0.88 +/- 0.03 |  

In [5]:
mlp_eval(X, y, reg_type=None, n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.79591835 0.81632656 0.73469388 0.875      0.72916669]
[0.73017544 0.79366829 0.73770343 0.87294477 0.73425214]
[0.78425588 0.81714559 0.73337645 0.87027483 0.72971154]
{'acc': 0.790221095085144, 'f1_macro': 0.7737488137137466, 'f1_weighted': 0.7869528559196068, 'acc_std': 0.05424449923276622, 'f1_macro_std': 0.054762249650224734, 'f1_weighted_std': 0.052930947915494395}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.75510204 0.85714287 0.79591835 0.9375     0.77083331]
[0.73028135 0.86745815 0.80430157 0.95072464 0.77475158]
[0.75822184 0.85827691 0.78980158 0.93683575 0.77207543]
{'acc': 0.8232993125915528, 'f1_macro': 0.8255034573952553, 'f1_weighted': 0.8230423024277783, 'acc_std': 0.06684377267029194, 'f1_macro_std': 0.07685449036354856, 'f1_weighted_std': 0.06647508594798308}
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.75510204 0.87755102 0.81632656 0.89583331 

| MLP no reg | 0.86 +/- 0.06 | 0.83 +/- 0.09 | 0.86 +/- 0.05 |  
| MLP no reg (mRNA only) | 0.87 +/- 0.04 | 0.85 +/- 0.05 | 0.87 +/- 0.04 |  
| MLP no reg (mRNA only) | 0.87 +/- 0.04 | 0.87 +/- 0.06 | 0.87 +/- 0.04 | 

In [6]:
mlp_eval(X, y, reg_type="l1", n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.77551019 0.85714287 0.81632656 0.91666669 0.77083331]
[0.72516835 0.85222222 0.82318841 0.93280632 0.77475158]
[0.77097506 0.85705215 0.81307306 0.91469038 0.77207543]
{'acc': 0.8272959232330322, 'f1_macro': 0.8216273766294975, 'f1_weighted': 0.8255732154725489, 'acc_std': 0.054530887642936364, 'f1_macro_std': 0.07042857909655777, 'f1_weighted_std': 0.05464799916236452}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 2 evals, cause 0.5276180169037312 < 0.8255732154725489
[0.6122449  0.65306121 0.63265306 0.         0.        ]
[0.37840909 0.47001263 0.41412338 0.         0.        ]
[0.48061224 0.57462379 0.51851312 0.         0.        ]
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 2 evals, cause 0.5413564643588856 < 0.8255732154725489
[0.65306121 0.63265306 0.67346936 0.         0.        ]
[0.475      0.42490222 0.50767544 0.         0.        ]
[0.56326531 0.51944762 0.5991

| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |  
| MLP l1 reg (mrna only) | 0.86 +/- 0.03 | 0.85 +/- 0.04 | 0.86 +/- 0.03 |  
| MLP l1 reg (mrna only) | 0.86 +/- 0.07 | 0.86 +/- 0.08 | 0.86 +/- 0.07 |  
| MLP l1 reg (mrna only, var) | 0.84 +/- 0.06 | 0.84 +/- 0.07 | 0.84 +/- 0.06 |


{'acc': 0.867463231086731, 'f1_macro': 0.8446657147200625, 'f1_weighted': 0.8687115181280782, 'acc_std': 0.0655361141104709, 'f1_macro_std': 0.09939143723563133, 'f1_weighted_std': 0.05932617979442903}

mrna only study.best_value=2.577811014967633, study.best_params={'lr': 0.00012898079159581874, 'l1_lambda': 0.00047366984296337946, 'l2_lambda': 0.0008509490658518463, 'proj_dim': 80, 'hidden_channels': 94}

In [7]:
mlp_eval(X, y, reg_type="inner_mat", n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.79591835 0.81632656 0.79591835 0.875      0.77083331]
[0.76975524 0.81948215 0.80430157 0.8947096  0.77475158]
[0.7956044  0.81951139 0.78980158 0.87082516 0.77207543]
{'acc': 0.8107993125915527, 'f1_macro': 0.8126000300289199, 'f1_weighted': 0.8095635923557335, 'acc_std': 0.035192175733458814, 'f1_macro_std': 0.04500308927522579, 'f1_weighted_std': 0.034183905959255544}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5


| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |  
| MLP (mrna only) | 0.87 +/- 0.06 | 0.87 +/- 0.07 | 0.87 +/- 0.06 |  
| MLP (mrna only) | 0.87 +/- 0.05 | 0.86 +/- 0.06 | 0.87 +/- 0.05 |  

{'acc': 0.8560374259948731, 'f1_macro': 0.8531890166223708, 'f1_weighted': 0.8540459975191783, 'acc_std': 0.05063744215757385, 'f1_macro_std': 0.06194529762727774, 'f1_weighted_std': 0.049910632703314174}

mrna only study.best_value=2.607328324144426, study.best_params={'lr': 8.077069025405151e-05, 'l1_lambda': 0.0004400516634407218, 'l2_lambda': 5.624746318975069e-05, 'proj_dim': 145, 'hidden_channels': 243}

# Results

| Method | ACC | F1 | F1 Weighted |
| --- | --- | --- | --- |
| KNN | 0.80 +/- 0.04 | 0.79 +/- 0.04 | 0.79 +/- 0.04 |
| LIN SVM w RFE | 0.84 +/- 0.01 | 0.84 +/- 0.02 | 0.84 +/- 0.01 |
| XGBoost | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |
| MLP no reg | 0.86 +/- 0.06 | 0.83 +/- 0.09 | 0.86 +/- 0.05 |
| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |
| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |
| MOGONET (mrna, cnv, mirna) | 0.766 ± 0.025 | 0.742 ± 0.027| 0.768 ± 0.024|
| MOGONET (mrna, meth, mirna) | 0.824 ± 0.029 | 0.812 ± 0.039| 0.824 ± 0.032|

In [55]:
# prepare 5 folds for cross validation
skf = StratifiedKFold(n_splits=5)

train_indices = []
val_indices = []
test_indices = []

for train_index, test_index in skf.split(X, y):
    train_indices.append(train_index)
    val_index, test_index = train_test_split(train_index, test_size=0.5, stratify=y[train_index])
    val_indices.append(val_index)
    test_indices.append(test_index)

train_idx = train_indices[0]
val_idx = val_indices[0]
test_idx = test_indices[0]

In [56]:
mrna_gene_names = mrna[:, 0].to_numpy()
meth_gene_names = meth[:, 0].to_numpy()
cna_gene_names = cnv[:, 0].to_numpy()
mirna_gene_names = mirna[:, 0].to_numpy()

mrna_mask = class_variational_selection(mrna_X[train_idx], y[train_idx], n_features=1000)
meth_mask = class_variational_selection(meth_X[train_idx], y[train_idx], n_features=1000)
cnv_mask = class_variational_selection(cnv_X[train_idx], y[train_idx], n_features=1000)
mirna_mask = class_variational_selection(mirna_X[train_idx], y[train_idx], n_features=200)

mrna_X_filt = torch.tensor(mrna_X[:, mrna_mask])
meth_X_filt = torch.tensor(meth_X[:, meth_mask])
cnv_X_filt = torch.tensor(cnv_X[:, cnv_mask])
mirna_X_filt = torch.tensor(mirna_X[:, mirna_mask])


mrna_gene_names = mrna_gene_names[mrna_mask]
meth_gene_names = meth_gene_names[meth_mask]
cna_gene_names = cna_gene_names[cnv_mask]
mirna_gene_names = mirna_gene_names[mirna_mask]

In [57]:
# build graph for mogonet
data = pyg.data.HeteroData()

# 0.95 produces a graph with mean degree of 10, this should be automatized later
eps = 0.95

# build adjacency matrix for each data type
A_mrna = cosine_similarity_matrix(mrna_X_filt)
A_mrna_bin = torch.where(A_mrna > eps, 1, 0)
A_mrna_bin.sum(axis=1, dtype=torch.float32).mean()

data['mrna'].x = mrna_X_filt
data['mrna'].edge_index = dense_to_coo(A_mrna_bin)
data.y = torch.tensor(y)

data.train_mask = torch.tensor(train_idx)
data.val_mask = torch.tensor(val_idx)
data.test_mask = torch.tensor(test_idx)

data

HeteroData(
  y=[483],
  train_mask=[386],
  val_mask=[193],
  test_mask=[193],
  mrna={
    x=[483, 1000],
    edge_index=[2, 4859],
  }
)

In [50]:
data.x_dict

{'mrna': tensor([[ 5.9121,  9.5405, 10.7415,  ..., 10.4321, 12.5581, 12.6258],
         [ 6.4948,  9.2530,  9.8974,  ...,  6.3334, 13.1286, 10.7483],
         [ 5.2825,  8.0867,  8.6480,  ...,  5.6983,  9.8676,  9.7985],
         ...,
         [ 0.7392,  7.5319,  5.3865,  ...,  3.0250,  6.2388,  3.9954],
         [ 7.2112,  9.0821,  8.8506,  ...,  6.6272,  3.7791,  9.2327],
         [ 5.6458,  9.2727,  8.6451,  ...,  9.6618, 10.2466, 11.6575]],
        dtype=torch.float64)}

In [51]:
data.edge_index_dict

{'mrna': tensor([[  0,   0,   0,  ..., 482, 482, 482],
         [  0,   1,   3,  ..., 453, 459, 482]])}

In [ ]:
model = ...

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

ccounts = torch.unique(data.y[train_mask], return_counts=True)[1]
inv_w = 1.0 / ccounts.float()
class_weights = inv_w / inv_w.sum()